# BirdCLEF+ 2026 - Pseudo-Label Generation

Use the best model (submission #5, EfficientNet-B0) to predict labels on
~10,592 unlabeled soundscape files. High-confidence predictions become
pseudo-labels for retraining.

**Required Kaggle Datasets:**
1. `birdclef-2026` - competition data (contains train_soundscapes)
2. `birdclef-baseline-model-v4` - best trained model
3. `birdclef-src` - source code

**Environment:** Kaggle GPU notebook (for speed)

In [ ]:
import os
import sys
import time

import numpy as np
import pandas as pd
import torch
import soundfile as sf
from tqdm.auto import tqdm

START_TIME = time.time()

ON_KAGGLE = os.path.exists("/kaggle/input")

if ON_KAGGLE:
    KAGGLE_USER = "stochastix"
    DATA_DIR = "/kaggle/input/competitions/birdclef-2026"
    MODEL_PATH = f"/kaggle/input/datasets/{KAGGLE_USER}/birdclef-baseline-model-v4/best_model.pth"
    SRC_DIR = f"/kaggle/input/datasets/{KAGGLE_USER}/birdclef-src"
    sys.path.insert(0, SRC_DIR)
else:
    DATA_DIR = "../data"
    MODEL_PATH = "../models/best_model.pth"
    sys.path.insert(0, "..")

SOUNDSCAPE_DIR = os.path.join(DATA_DIR, "train_soundscapes")
LABELS_CSV = os.path.join(DATA_DIR, "train_soundscapes_labels.csv")
TAXONOMY_CSV = os.path.join(DATA_DIR, "taxonomy.csv")
OUTPUT_DIR = "/kaggle/working" if ON_KAGGLE else "../data"

print(f"ON_KAGGLE: {ON_KAGGLE}")
print(f"MODEL_PATH: {MODEL_PATH}")
print(f"Model exists: {os.path.exists(MODEL_PATH)}")

In [ ]:
import yaml
from src.audio import load_audio, audio_to_melspec, normalize_melspec
from src.transforms import spectrogram_to_tensor
from src.dataset import load_taxonomy
from src.model import get_model

# Load config
config_path = os.path.join(SRC_DIR, "config", "default.yaml") if ON_KAGGLE else "../config/default.yaml"
with open(config_path) as f:
    config = yaml.safe_load(f)

audio_config = config["audio"]
SR = audio_config["sample_rate"]
DURATION = audio_config["duration"]
IN_CHANNELS = config["model"].get("in_channels", 1)

# Load species list
species_list = load_taxonomy(TAXONOMY_CSV)
print(f"Species: {len(species_list)}")

## Load Model

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

config["model"]["pretrained"] = False
model = get_model(config["model"])
model.load_state_dict(torch.load(MODEL_PATH, map_location=device, weights_only=True))
model.to(device)
model.eval()

params = sum(p.numel() for p in model.parameters())
print(f"Model loaded: {params:,} parameters")
print(f"Time elapsed: {time.time() - START_TIME:.1f}s")

## Identify Unlabeled Soundscapes

In [ ]:
# Load real labels to identify which files are already labeled
real_labels = pd.read_csv(LABELS_CSV)
labeled_files = set(real_labels["filename"].unique())

# Get all soundscape files
all_files = sorted([f for f in os.listdir(SOUNDSCAPE_DIR) if f.endswith(".ogg")])
unlabeled_files = [f for f in all_files if f not in labeled_files]

print(f"Total soundscape files: {len(all_files)}")
print(f"Labeled files: {len(labeled_files)}")
print(f"Unlabeled files: {len(unlabeled_files)}")
print(f"Time elapsed: {time.time() - START_TIME:.1f}s")

## Run Inference on Unlabeled Soundscapes

In [ ]:
# Configuration
CONFIDENCE_THRESHOLD = 0.5  # keep predictions above this
BATCH_SIZE = 64  # GPU batch size for speed

def make_spectrogram(audio, sr, audio_config, in_channels):
    """Convert audio to model-ready tensor."""
    mel = audio_to_melspec(audio, sr, audio_config)
    mel = normalize_melspec(mel)
    return spectrogram_to_tensor(mel, in_channels)

# Collect pseudo-labels
pseudo_rows = []
total_windows = 0
total_positive_windows = 0

for file_idx, filename in enumerate(tqdm(unlabeled_files, desc="Pseudo-labeling")):
    file_path = os.path.join(SOUNDSCAPE_DIR, filename)
    
    # Get file duration from header (no decoding)
    info = sf.info(file_path)
    total_dur = info.duration
    
    # Build windows for this file
    windows = []
    start = 0.0
    while start + DURATION <= total_dur + 0.01:
        windows.append(start)
        start += DURATION
    
    if not windows:
        continue
    
    # Load audio once for the entire file
    audio_full = load_audio(file_path, sr=SR, duration=total_dur, offset=0.0)
    
    # Create spectrograms for all windows
    batch_specs = []
    for win_start in windows:
        start_sample = int(win_start * SR)
        end_sample = start_sample + int(DURATION * SR)
        audio_chunk = audio_full[start_sample:end_sample]
        
        # Pad if needed
        if len(audio_chunk) < int(DURATION * SR):
            audio_chunk = np.pad(audio_chunk, (0, int(DURATION * SR) - len(audio_chunk)))
        
        spec = make_spectrogram(audio_chunk, SR, audio_config, IN_CHANNELS)
        batch_specs.append(spec)
    
    # Process in batches
    all_probs = []
    for i in range(0, len(batch_specs), BATCH_SIZE):
        batch = torch.stack(batch_specs[i:i + BATCH_SIZE]).to(device)
        with torch.no_grad():
            output = model(batch)
            logits = output["clipwise_logits"] if isinstance(output, dict) else output
            probs = torch.sigmoid(logits).cpu().numpy()
        all_probs.append(probs)
    
    all_probs = np.concatenate(all_probs, axis=0)  # (num_windows, 234)
    total_windows += len(windows)
    
    # Convert high-confidence predictions to labels
    for win_idx, win_start in enumerate(windows):
        win_end = win_start + DURATION
        probs = all_probs[win_idx]
        
        # Get species above threshold
        confident_species = [species_list[j] for j in range(len(species_list)) if probs[j] >= CONFIDENCE_THRESHOLD]
        
        if confident_species:
            total_positive_windows += 1
            pseudo_rows.append({
                "filename": filename,
                "start": win_start,
                "end": win_end,
                "primary_label": ";".join(confident_species),
            })
    
    # Progress update every 500 files
    if (file_idx + 1) % 500 == 0:
        print(f"  Processed {file_idx + 1}/{len(unlabeled_files)} files, "
              f"{total_positive_windows}/{total_windows} positive windows so far")

print(f"\nDone! Processed {len(unlabeled_files)} files")
print(f"Total windows: {total_windows}")
print(f"Positive windows (>= {CONFIDENCE_THRESHOLD} confidence): {total_positive_windows}")
print(f"Positive rate: {total_positive_windows/max(total_windows, 1)*100:.1f}%")
print(f"Time elapsed: {time.time() - START_TIME:.1f}s")

## Analyze Pseudo-Labels

In [ ]:
pseudo_df = pd.DataFrame(pseudo_rows)
print(f"Pseudo-label rows: {len(pseudo_df)}")
print(f"Files with pseudo-labels: {pseudo_df['filename'].nunique()}")

# Count species occurrences in pseudo-labels
all_species_counts = {}
for labels_str in pseudo_df["primary_label"]:
    for sp in labels_str.split(";"):
        all_species_counts[sp] = all_species_counts.get(sp, 0) + 1

print(f"\nUnique species in pseudo-labels: {len(all_species_counts)}")
print(f"\nTop 20 most frequent species:")
for sp, count in sorted(all_species_counts.items(), key=lambda x: -x[1])[:20]:
    print(f"  {sp}: {count}")

# Compare with real labels
real_species = set()
for labels_str in real_labels["primary_label"]:
    for sp in str(labels_str).split(";"):
        if sp in species_list:
            real_species.add(sp)

pseudo_species = set(all_species_counts.keys())
new_species = pseudo_species - real_species
print(f"\nSpecies in real labels: {len(real_species)}")
print(f"Species in pseudo-labels: {len(pseudo_species)}")
print(f"New species from pseudo-labels (not in real soundscape labels): {len(new_species)}")
if new_species:
    print(f"  Examples: {list(new_species)[:10]}")

## Save Pseudo-Labels

In [ ]:
# Save pseudo-labels CSV
output_path = os.path.join(OUTPUT_DIR, "pseudo_labels.csv")
pseudo_df.to_csv(output_path, index=False)
print(f"Saved pseudo-labels to: {output_path}")
print(f"File size: {os.path.getsize(output_path) / 1024:.1f} KB")

# Also save combined labels (real + pseudo) for convenience
combined_df = pd.concat([real_labels, pseudo_df], ignore_index=True)
combined_path = os.path.join(OUTPUT_DIR, "combined_soundscape_labels.csv")
combined_df.to_csv(combined_path, index=False)
print(f"\nSaved combined labels to: {combined_path}")
print(f"  Real label rows: {len(real_labels)}")
print(f"  Pseudo-label rows: {len(pseudo_df)}")
print(f"  Combined rows: {len(combined_df)}")

print(f"\nTotal runtime: {time.time() - START_TIME:.1f}s ({(time.time() - START_TIME)/60:.1f} min)")

## Next Steps

1. Download `pseudo_labels.csv` (or `combined_soundscape_labels.csv`) from this notebook's output
2. Upload as a Kaggle dataset (e.g., `birdclef-pseudo-labels`)
3. In the training notebook, load the combined labels instead of just real labels:
   ```python
   # Instead of:
   soundscape_labels = pd.read_csv(soundscape_csv)
   # Use:
   soundscape_labels = pd.read_csv(combined_labels_csv)
   ```
4. Retrain and submit